# Notebook 03 — Variance scaling and the $\sqrt{d_k}$ correction

Paired with `theory/03_why_sqrt_dk.md`. **Mode:** theory-first — read the chapter end-to-end and write your §3.7 predictions before running anything.

## Question, Hypothesis, Method, Prediction, Confidence

**Question.** Does Lemma 3.1 empirically hold for i.i.d. Gaussian $q, k$ across $d_k \in \{1, 4, 16, 64, 256, 1024\}$? Does the $1/\sqrt{d_k}$ correction restore unit variance? At what $d_k$ does unscaled attention functionally break — softmax saturates, Jacobian vanishes?

**Hypothesis.** Lemma 3.1 and Proposition 3.2 hold exactly in expectation; Monte-Carlo variance of the estimator is $O(1/\sqrt{M})$ in the sample count $M$. Softmax entropy and Jacobian norm should show a sharp transition around the $d_k$ where score std exceeds $\sim 3$ — i.e. $d_k \approx 9$.

**Method.** Three experiments on i.i.d. $\mathcal{N}(0, 1)$ $q, k$ with no trained model.
1. Empirical score-variance curve (scaled vs. unscaled) across $d_k$, with $M = 10^5$ Monte-Carlo samples.
2. Softmax output statistics (entropy, max-to-mean ratio) on single-query rows, $n = 32$ keys, 1000 trials per $d_k$.
3. Softmax Jacobian Frobenius norm on the same sampled score vectors.

**Prediction.** *Fill this in before running anything — reference your §3.7 predictions. For each of the five predictions in §3.7 state: direction, order-of-magnitude estimate, confidence.*

1. Unscaled score variance vs. $d_k$ — 
2. Scaled score variance vs. $d_k$ — 
3. Softmax entropy curves (unscaled vs. scaled) — 
4. Max-to-mean ratio in a softmax row — 
5. Jacobian Frobenius norm decay — 

**Confidence.** Overall: *[low / medium / high, with a one-line reason]*.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

D_KS = [1, 4, 16, 64, 256, 1024]
M = 100_000         # Monte-Carlo sample count for variance estimation
N_KEYS = 32         # keys per query for softmax experiments
N_TRIALS = 1000     # query samples per d_k for softmax / Jacobian experiments

---
## Experiment 1 — empirical score variance

Verifying Lemma 3.1: $\mathrm{Var}(\langle q, k \rangle) = d_k$ for i.i.d. unit-variance entries, and Proposition 3.2: $\mathrm{Var}(\langle q, k \rangle / \sqrt{d_k}) = 1$.

In [ ]:
var_unscaled = []
var_scaled = []

for d_k in D_KS:
    q = np.random.randn(M, d_k)
    k = np.random.randn(M, d_k)
    s = np.sum(q * k, axis=1)
    var_unscaled.append(s.var())
    var_scaled.append((s / np.sqrt(d_k)).var())
    print(f'd_k={d_k:>5}   Var(s) = {s.var():10.3f}   Var(s/sqrt(d_k)) = {(s / np.sqrt(d_k)).var():6.4f}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(D_KS, var_unscaled, 'o-', label='unscaled')
ax.plot(D_KS, var_scaled, 's-', label='scaled by 1/sqrt(d_k)')
ax.plot(D_KS, D_KS, ':', alpha=0.5, label='y = d_k (Lemma 3.1)')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('d_k')
ax.set_ylabel('empirical Var(score)')
ax.set_title('Score variance vs head dimension')
ax.legend()
plt.show()

**Sub-finding 1.** *Write 1–3 sentences. Did the unscaled curve land on $y = d_k$? Did the scaled curve hold at 1? Any deviation worth noting?*

---
## Experiment 2 — softmax output statistics

For each $d_k$ we draw `N_TRIALS` independent $(q, K)$ pairs with $n = 32$ keys, compute softmax of the attention scores, and measure two summary statistics:

- **Entropy** of the softmax row. Max is $\log n \approx 3.47$ (uniform); min is $0$ (one-hot).
- **Max-to-mean ratio** in the row. Uniform gives ratio $1$; one-hot gives ratio $n$.

In [ ]:
def softmax(x):
    x = x - x.max(axis=-1, keepdims=True)
    ex = np.exp(x)
    return ex / ex.sum(axis=-1, keepdims=True)

entropy_unscaled, entropy_scaled = [], []
max_ratio_unscaled, max_ratio_scaled = [], []

for d_k in D_KS:
    q = np.random.randn(N_TRIALS, d_k)
    K = np.random.randn(N_TRIALS, N_KEYS, d_k)
    scores = np.einsum('td,tnd->tn', q, K)
    for scaled, H_store, R_store in [
        (False, entropy_unscaled, max_ratio_unscaled),
        (True,  entropy_scaled,   max_ratio_scaled),
    ]:
        s = scores / np.sqrt(d_k) if scaled else scores
        p = softmax(s)
        H = -np.sum(p * np.log(p + 1e-30), axis=-1).mean()
        R = (p.max(axis=-1) / p.mean(axis=-1)).mean()
        H_store.append(H)
        R_store.append(R)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(D_KS, entropy_unscaled, 'o-', label='unscaled')
axes[0].plot(D_KS, entropy_scaled,   's-', label='scaled')
axes[0].axhline(np.log(N_KEYS), color='k', linestyle=':', alpha=0.5, label=f'log(n) = log({N_KEYS})')
axes[0].set_xscale('log')
axes[0].set_xlabel('d_k')
axes[0].set_ylabel('mean softmax entropy (nats)')
axes[0].set_title('Softmax entropy')
axes[0].legend()

axes[1].plot(D_KS, max_ratio_unscaled, 'o-', label='unscaled')
axes[1].plot(D_KS, max_ratio_scaled,   's-', label='scaled')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlabel('d_k')
axes[1].set_ylabel('max / mean in softmax row')
axes[1].set_title('Softmax row concentration')
axes[1].legend()

plt.tight_layout()
plt.show()

**Sub-finding 2.** *Did the unscaled entropy collapse toward $0$? At what $d_k$ did the collapse begin, operationally? The scaled case saturates slightly below $\log n$ — where does the gap come from?*

---
## Experiment 3 — Jacobian Frobenius norm (Lemma 3.3)

For each $d_k$ we evaluate $\|J\|_F$ where $J_{ij} = p_i(\delta_{ij} - p_j)$, $p = \mathrm{softmax}(s)$, averaged over random score vectors. Lemma 3.3 predicts $\|J\|_F \to 0$ under saturation. We expect sharp decay in the unscaled case past a threshold $d_k$.

In [ ]:
def softmax_jacobian_frobenius(p):
    # ||J||_F^2 = sum_i p_i^2 (1 - p_i)^2  +  sum_{i != j} p_i^2 p_j^2
    P2 = p ** 2
    diag = (P2 * (1 - p) ** 2).sum(axis=-1)
    off = P2.sum(axis=-1) ** 2 - (P2 ** 2).sum(axis=-1)
    return np.sqrt(diag + off)

jac_unscaled, jac_scaled = [], []

for d_k in D_KS:
    q = np.random.randn(N_TRIALS, d_k)
    K = np.random.randn(N_TRIALS, N_KEYS, d_k)
    scores = np.einsum('td,tnd->tn', q, K)
    for scaled, store in [(False, jac_unscaled), (True, jac_scaled)]:
        s = scores / np.sqrt(d_k) if scaled else scores
        p = softmax(s)
        store.append(softmax_jacobian_frobenius(p).mean())

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(D_KS, jac_unscaled, 'o-', label='unscaled')
ax.plot(D_KS, jac_scaled,   's-', label='scaled')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('d_k')
ax.set_ylabel('mean ||J||_F')
ax.set_title('Softmax Jacobian Frobenius norm')
ax.legend()
plt.show()

**Sub-finding 3.** *Did the unscaled Jacobian norm decay polynomially or exponentially in $d_k$? (Re-plot with linear axes to decide.) Is the scaled curve flat? At what $d_k$ do the scaled and unscaled curves visibly diverge, and does that match the threshold from Experiment 2?*

---
## Finding

*Write 3–6 sentences answering:*

- Which predictions held?
- Which missed, and in what direction? State the update to the mental model in one sentence.
- Any surprises (numerical precision effects, nonmonotone behavior, finite-sample artifacts)?
- What's the next question this raises? Candidate: does Adam's preconditioning offset the vanishing Jacobian enough to let unscaled attention train at moderate $d_k$? That is Exercise 3.4 and a future experiment.

*Append a 2–3 sentence summary to `learnings.md` with today's date and a link back to this notebook.*

---
## Optional extension — non-Gaussian entries

Lemma 3.1 depends only on the first two moments of $q_i, k_i$. Repeat Experiment 1 with entries drawn from (a) Uniform$[-\sqrt{3}, \sqrt{3}]$ and (b) Laplace$(0, 1/\sqrt{2})$ — both unit variance, neither Gaussian. The variance curves should land on the same line. This is a 5-line check that strengthens confidence in the mechanism, not in a specific distribution.

In [ ]:
def sample(dist, shape):
    if dist == 'gaussian':
        return np.random.randn(*shape)
    if dist == 'uniform':
        return np.random.uniform(-np.sqrt(3), np.sqrt(3), size=shape)
    if dist == 'laplace':
        return np.random.laplace(loc=0.0, scale=1.0 / np.sqrt(2), size=shape)
    raise ValueError(dist)

fig, ax = plt.subplots(figsize=(6, 4))
for dist in ['gaussian', 'uniform', 'laplace']:
    vars_ = []
    for d_k in D_KS:
        q = sample(dist, (M, d_k))
        k = sample(dist, (M, d_k))
        vars_.append(np.sum(q * k, axis=1).var())
    ax.plot(D_KS, vars_, 'o-', label=dist)
ax.plot(D_KS, D_KS, ':', alpha=0.5, label='y = d_k')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('d_k')
ax.set_ylabel('Var(score), unscaled')
ax.set_title('Lemma 3.1 is distribution-free given first two moments')
ax.legend()
plt.show()